# VolleyScout — Phases 0–5 (train on `train/`, test on `holdout/`)

**Ubuntu + RTX 3050 (6 GB VRAM)** — Jupyter kernel: `py312`

| Phase | What |
|-------|------|
| **0** | Ground-truth template + evaluation metrics |
| **1** | Fine-tuned **volleyball ball** YOLO + court ROI + Kalman tracking |
| **2** | Rally / event logic (multi-frame contacts, Point, real time windows) |
| **3** | Jersey OCR crops + temporal voting |
| **4** | **BoT-SORT** player tracking, GPU, `infer_stride=1` |
| **5** | Smoke test (90s holdout) + holdout batch + evaluation JSON |

| Component | Method |
|-----------|--------|
| Player detection | YOLO11 → `models/volleyball_player_best.pt` |
| Ball detection | YOLO11 → `models/volleyball_ball_best.pt` |
| Tracking | **BoT-SORT** |
| Jersey | PaddleOCR / EasyOCR on torso crops |
| Events | `volleyball_analytics/pipeline.py` |

### Run order
1. **§1** Install → **Restart kernel**
2. **§2** Config → **§2b** Fresh start (clears old `output`, `dataset`, `frames`, `models`, `runs`)
3. **§3** Video inventory (`train/` vs `holdout/`)
4. **§4–§5** Player frames + pseudo-labels → **§10–§11** player train/eval
5. **§4b–§5b** Ball frames + pseudo-labels → **§10b–§11b** ball train/eval
6. **§12** Pipeline → **§13** Smoke (first **holdout** video) → **§14** All holdout smoke/eval
7. Optional **§15** upload / full export

> **Testing policy:** always evaluate on **`holdout/`** videos only after training on **`train/`**.


## 1. Environment setup
Install deps once, then restart the kernel if prompted.


In [2]:
# One-time install for conda env py312 (Ubuntu + CUDA). Restart kernel after this cell.
!nvidia-smi
# %pip install -q -r requirements-analytics.txt
print('Done. Kernel -> Restart, then run imports cell.')


Wed Oct  7 17:43:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.91.07              Driver Version: 595.91.07      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050        Off |   00000000:01:00.0  On |                  N/A |
|  0%   57C    P5             14W /   70W |    5345MiB /   6144MiB |     34%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os, re, json, random, shutil, time, warnings
from pathlib import Path
from collections import defaultdict, deque

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import torch
from ultralytics import YOLO

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


PyTorch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 3050
VRAM (GB): 6.09


## 2. Project configuration (local paths)

Edit only this cell if your folder layout changes.


In [4]:
# =========================
# LOCAL PROJECT ROOT (auto)
# =========================
PROJECT_ROOT = Path.cwd().resolve()
# This notebook may be opened from a subfolder. Find the repository by its
# pipeline package; never fall back to a machine-specific path.
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / 'volleyball_analytics' / 'pipeline.py').exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError('Open this notebook from the VolleyScout repository.')

TRAIN_VIDEO_DIR = PROJECT_ROOT / 'train'
TEST_VIDEO_DIR  = PROJECT_ROOT / 'holdout'

FRAMES_DIR   = PROJECT_ROOT / 'frames'
DATASET_ROOT = PROJECT_ROOT / 'dataset'
RUNS_DIR     = PROJECT_ROOT / 'runs'
MODELS_DIR   = PROJECT_ROOT / 'models'
OUTPUT_DIR   = PROJECT_ROOT / 'output'
EVENTS_DIR   = OUTPUT_DIR / 'events_csv'
CROPS_DIR    = OUTPUT_DIR / 'jersey_crops'
UPLOAD_DIR   = PROJECT_ROOT / 'uploads'

TRAIN_IMAGES = DATASET_ROOT / 'images' / 'train'
VAL_IMAGES   = DATASET_ROOT / 'images' / 'val'
TEST_IMAGES  = DATASET_ROOT / 'images' / 'test'
TRAIN_LABELS = DATASET_ROOT / 'labels' / 'train'
VAL_LABELS   = DATASET_ROOT / 'labels' / 'val'
TEST_LABELS  = DATASET_ROOT / 'labels' / 'test'

for p in [FRAMES_DIR, DATASET_ROOT, RUNS_DIR, MODELS_DIR, OUTPUT_DIR,
          EVENTS_DIR, CROPS_DIR, UPLOAD_DIR, TRAIN_IMAGES, VAL_IMAGES, TEST_IMAGES,
          TRAIN_LABELS, VAL_LABELS, TEST_LABELS]:
    p.mkdir(parents=True, exist_ok=True)

PRETRAINED_MODEL = 'yolo11n.pt'
BALL_MODEL       = 'yolo11n.pt'
EPOCHS, IMAGE_SIZE, BATCH_SIZE = 40, 640, 4
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
CONF_DET, CONF_PSEUDO = 0.35, 0.55
FRAME_INTERVAL, MAX_FRAMES_PER_VIDEO, PERSON_CLASS_ID = 30, 250, 0
SMOKE_MAX_SECONDS = 90

LOCAL_BEST = MODELS_DIR / 'volleyball_player_best.pt'
RUN_BEST   = RUNS_DIR / 'volleyball_player_detection' / 'weights' / 'best.pt'
EVENT_NAMES = ['Serve', 'Reception', 'Set', 'Attack', 'Block', 'Dig', 'Rally', 'Point']
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('OUTPUT (output):', OUTPUT_DIR)

BALL_DATASET_ROOT = PROJECT_ROOT / 'dataset_ball'
BALL_TRAIN_IMAGES = BALL_DATASET_ROOT / 'images' / 'train'
BALL_VAL_IMAGES   = BALL_DATASET_ROOT / 'images' / 'val'
BALL_TEST_IMAGES  = BALL_DATASET_ROOT / 'images' / 'test'
BALL_TRAIN_LABELS = BALL_DATASET_ROOT / 'labels' / 'train'
BALL_VAL_LABELS   = BALL_DATASET_ROOT / 'labels' / 'val'
BALL_TEST_LABELS  = BALL_DATASET_ROOT / 'labels' / 'test'
GROUND_TRUTH_DIR  = PROJECT_ROOT / 'ground_truth'

BALL_FRAME_INTERVAL, BALL_CONF_PSEUDO, BALL_EPOCHS = 15, 0.22, 30
SPORTS_BALL_CLASS = 32
LOCAL_BALL_BEST = MODELS_DIR / 'volleyball_ball_best.pt'
RUN_BALL_BEST   = RUNS_DIR / 'volleyball_ball_detection' / 'weights' / 'best.pt'
DATA_BALL_YAML  = PROJECT_ROOT / 'data_ball.yaml'
FRESH_START = False

for p in [BALL_DATASET_ROOT, BALL_TRAIN_IMAGES, BALL_VAL_IMAGES, BALL_TEST_IMAGES,
          BALL_TRAIN_LABELS, BALL_VAL_LABELS, BALL_TEST_LABELS, GROUND_TRUTH_DIR]:
    p.mkdir(parents=True, exist_ok=True)


PROJECT_ROOT: /media/iflair/New Volume/VolleyScout
OUTPUT (output): /media/iflair/New Volume/VolleyScout/output


## 2b. Fresh start — clear old artifacts

Run once when re-training from scratch (`FRESH_START = True` in §2).  
Removes previous **`output`**, **`dataset`**, **`dataset_ball`**, **`frames`**, **`models/*.pt`**, and **`runs/`** (keeps `train/` and `holdout/` videos).


In [5]:
def _rm_tree(path: Path):
    if not path.exists():
        return
    if path.is_file():
        path.unlink()
        return
    for child in path.iterdir():
        _rm_tree(child)
    path.rmdir()

def fresh_start_cleanup():
    if not FRESH_START:
        print('FRESH_START is False — skipping cleanup.')
        return
    print('=== Fresh start cleanup ===')
    for d in [OUTPUT_DIR, DATASET_ROOT, BALL_DATASET_ROOT, FRAMES_DIR, RUNS_DIR]:
        if d.exists():
            for child in list(d.iterdir()):
                _rm_tree(child)
            print(' cleared', d)
    for f in MODELS_DIR.glob('*.pt'):
        f.unlink()
        print(' removed', f)
    for sub in [EVENTS_DIR, CROPS_DIR]:
        sub.mkdir(parents=True, exist_ok=True)
    print('Done. Re-run §4 onward to rebuild dataset and models.')

fresh_start_cleanup()


FRESH_START is False — skipping cleanup.


## 3. Video inventory & train / holdout check

Lists videos under `train/` and `holdout/`. **Equal counts are recommended but not required.**
If both folders are empty, you will see a message — add match files or use **§14 upload** later.


In [6]:
from IPython.display import display

def list_videos(folder: Path):
    folder = Path(folder)
    if not folder.exists():
        return []
    exts = {e.lower() for e in VIDEO_EXTS}
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in exts
    )

def video_metadata(video_path: Path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        size_gb = round(video_path.stat().st_size / 1e9, 2) if video_path.exists() else 0
        return {
            "video": video_path.name, "path": str(video_path),
            "ok": False, "fps": 0, "frames": 0, "width": 0, "height": 0,
            "duration_min": 0, "size_gb": size_gb,
        }
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    cap.release()
    ok = frames > 0 and fps > 0
    dur = (frames / fps / 60.0) if fps else 0
    return {
        "video": video_path.name, "path": str(video_path), "ok": ok,
        "fps": round(fps, 2), "frames": frames, "width": width, "height": height,
        "duration_min": round(dur, 1),
        "size_gb": round(video_path.stat().st_size / 1e9, 2),
    }

train_vids = list_videos(TRAIN_VIDEO_DIR)
test_vids  = list_videos(TEST_VIDEO_DIR)

print(f"TRAIN dir: {TRAIN_VIDEO_DIR}")
print(f"HOLDOUT dir: {TEST_VIDEO_DIR}")
print(f"TRAIN videos: {len(train_vids)}")
print(f"TEST  videos (holdout): {len(test_vids)}")

if len(train_vids) == 0 and len(test_vids) == 0:
    print("\nNo videos found yet. Copy match .mp4/.MP4 files into train/ and holdout/.")
elif len(train_vids) != len(test_vids):
    print(
        f"\nNOTE: train/holdout counts differ ({len(train_vids)} vs {len(test_vids)}). "
        "This is OK — pipeline continues."
    )
else:
    print("Equal ratio OK:", len(train_vids), "/", len(test_vids))

readable_train = []
broken_train = []
for v in train_vids:
    m = video_metadata(v)
    (readable_train if m["ok"] else broken_train).append(v)

if broken_train:
    print("\nWARNING - unreadable train videos (skipped):")
    for v in broken_train:
        print(" ", v.name)

# Val split: last 2 train matches when >=3; last 1 when ==2; none when only 1 train video
if len(readable_train) >= 3:
    VAL_VIDEOS = readable_train[-2:]
elif len(readable_train) == 2:
    VAL_VIDEOS = readable_train[-1:]
else:
    VAL_VIDEOS = []

TRAIN_VIDEOS = [v for v in readable_train if v not in VAL_VIDEOS]

readable_test = []
broken_test = []
for v in test_vids:
    m = video_metadata(v)
    (readable_test if m["ok"] else broken_test).append(v)
if broken_test:
    print("\nWARNING - unreadable holdout videos (skipped):")
    for v in broken_test:
        print(" ", v.name)
TEST_VIDEOS = readable_test

print("\nEffective split (video-level):")
print(f"  train frames from: {len(TRAIN_VIDEOS)} videos")
print(f"  val   frames from: {len(VAL_VIDEOS)} videos")
print(f"  test  / unseen   : {len(TEST_VIDEOS)} videos")

records = []
for split, vids in [("train", TRAIN_VIDEOS), ("val", VAL_VIDEOS), ("test", TEST_VIDEOS)]:
    for v in vids:
        m = video_metadata(v)
        m["split"] = split
        records.append(m)

cols = ["split", "video", "width", "height", "fps", "duration_min", "size_gb", "ok"]
video_df = pd.DataFrame(records, columns=cols if not records else None)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
video_df.to_csv(OUTPUT_DIR / "video_inventory.csv", index=False)
print("Saved:", OUTPUT_DIR / "video_inventory.csv")

if video_df.empty:
    display(video_df)
else:
    display(video_df[cols])



def readable_videos(folder):
    return [v for v in list_videos(folder) if video_metadata(v)['ok']]


TRAIN dir: C:\Users\sarth\Music\VolleyScout\train
HOLDOUT dir: C:\Users\sarth\Music\VolleyScout\holdout
TRAIN videos: 10
TEST  videos (holdout): 10
Equal ratio OK: 10 / 10

WARNING - unreadable train videos (skipped):
  2025#10#11#MODENA VOLLEY MO#TEAM VOLLEY CAZZAGO ADRO BS#Serie BM B#01_Modena_bmb_25.MP4

Effective split (video-level):
  train frames from: 7 videos
  val   frames from: 2 videos
  test  / unseen   : 10 videos
Saved: C:\Users\sarth\Music\VolleyScout\output\video_inventory.csv


,split,video,width,height,fps,duration_min,size_gb,ok
0,train,2025#09#23#VERO VOLLEY MONZA MB#ASD PALLAVOLO ...,1280,720,30.0,144.3,2.16,True
1,train,2025#09#25#NTRANSPORTS VILLADORO MO#MORE VOLLE...,1280,720,25.0,93.0,2.18,True
2,train,2025#09#26#CANOTTIERI ONGINA PC#ASD PALLAVOLO ...,1280,720,30.0,99.6,1.54,True
3,train,2025#10#11#MOTTA&ROSSI ASOLA REMEDELLO MN#VERO...,1280,720,30.0,149.6,2.26,True
4,train,2025#10#11#PALLAV CISANO BSCO SSDARL BG#CANOTT...,720,576,25.0,124.8,0.94,True
5,train,2025#10#11#PALLAV CISANO BSCO SSDARL BG#CANOTT...,720,576,25.0,126.2,1.22,True
6,train,2025#10#11#ZOTUP SCANZOROSCIATE BG#BECA TENSPE...,1280,720,30.0,106.8,1.65,True
7,val,2025#10#18#CANOTTIERI ONGINA PC#FERRAMENTA AST...,720,576,25.0,106.1,1.02,True
8,val,2025#10#18#KERAKOLL SASSUOLO MO#MOTTA&ROSSI AS...,1280,720,30.0,112.7,1.70,True
9,test,2025#09#11#KERAKOLL SASSUOLO MO#NTRANSPORTS VI...,1280,720,25.0,71.0,1.67,True


## 3b. Phase 0 — ground truth template (optional)

Fill `ground_truth/<video_stem>_events.csv` to score event timing in §13/§13b (`event`, `time_start_sec`).


In [7]:
template = pd.DataFrame([
    {'event': 'Serve', 'time_start_sec': 0.0, 'notes': 'edit me'},
    {'event': 'Reception', 'time_start_sec': 1.0, 'notes': ''},
    {'event': 'Point', 'time_start_sec': 8.0, 'notes': 'rally end'},
])
example = GROUND_TRUTH_DIR / 'README_ground_truth.csv'
template.to_csv(example, index=False)
print('Template:', example)
print('Copy to ground_truth/<holdout_stem>_events.csv and edit timestamps after watching video.')


Template: C:\Users\sarth\Music\VolleyScout\ground_truth\README_ground_truth.csv
Copy to ground_truth/<holdout_stem>_events.csv and edit timestamps after watching video.


## 3b. Phase 0 — ground truth template (optional)

Fill `ground_truth/<video_stem>_events.csv` to score event timing in §13/§13b (`event`, `time_start_sec`).


In [8]:
template = pd.DataFrame([
    {'event': 'Serve', 'time_start_sec': 0.0, 'notes': 'edit me'},
    {'event': 'Reception', 'time_start_sec': 1.0, 'notes': ''},
    {'event': 'Point', 'time_start_sec': 8.0, 'notes': 'rally end'},
])
example = GROUND_TRUTH_DIR / 'README_ground_truth.csv'
template.to_csv(example, index=False)
print('Template:', example)
print('Copy to ground_truth/<holdout_stem>_events.csv and edit timestamps after watching video.')


Template: C:\Users\sarth\Music\VolleyScout\ground_truth\README_ground_truth.csv
Copy to ground_truth/<holdout_stem>_events.csv and edit timestamps after watching video.


## 4. Frame extraction (sampled, not every frame)

Extracts frames for the YOLO dataset. Full-match inference later writes complete videos to `output`.

In [9]:
def extract_frames(video_path, output_dir, frame_interval=FRAME_INTERVAL, max_frames=MAX_FRAMES_PER_VIDEO):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"SKIP (cannot open): {video_path.name}")
        return 0
    frame_idx, saved = 0, 0
    stem = re.sub(r"[^\w\-]+", "_", video_path.stem)[:80]
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_idx % frame_interval == 0:
            out = output_dir / f"{stem}_f{frame_idx:08d}.jpg"
            h, w = frame.shape[:2]
            if w > 1280:
                frame = cv2.resize(frame, (1280, int(h * 1280 / w)))
            cv2.imwrite(str(out), frame, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
            saved += 1
            if max_frames is not None and saved >= max_frames:
                break
        frame_idx += 1
    cap.release()
    return saved

# Extract into frames/{split}/{video_stem}/
extract_plan = [
    ("train", TRAIN_VIDEOS),
    ("val", VAL_VIDEOS),
    ("test", TEST_VIDEOS),  # all holdout matches
]

extraction_log = []
for split, vids in extract_plan:
    for v in vids:
        out_dir = FRAMES_DIR / split / re.sub(r"[^\w\-]+", "_", v.stem)[:80]
        n = extract_frames(v, out_dir)
        # n = extract_frames(v, out_dir, max_frames=None if split == "test" else MAX_FRAMES_PER_VIDEO)
        extraction_log.append({"split": split, "video": v.name, "frames_saved": n, "out": str(out_dir)})
        print(f"[{split}] {v.name[:55]}... -> {n} frames")

pd.DataFrame(extraction_log).to_csv(OUTPUT_DIR / "frame_extraction_log.csv", index=False)
print("\nTotal extracted:", sum(r["frames_saved"] for r in extraction_log))


[train] 2025#09#23#VERO VOLLEY MONZA MB#ASD PALLAVOLO SARONNO V... -> 250 frames
[train] 2025#09#25#NTRANSPORTS VILLADORO MO#MORE VOLLEY MO#AMIC... -> 250 frames
[train] 2025#09#26#CANOTTIERI ONGINA PC#ASD PALLAVOLO CREMONESE... -> 250 frames
[train] 2025#10#11#MOTTA&ROSSI ASOLA REMEDELLO MN#VERO VOLLEY M... -> 250 frames
[train] 2025#10#11#PALLAV CISANO BSCO SSDARL BG#CANOTTIERI ONGI... -> 250 frames
[train] 2025#10#11#PALLAV CISANO BSCO SSDARL BG#CANOTTIERI ONGI... -> 250 frames
[train] 2025#10#11#ZOTUP SCANZOROSCIATE BG#BECA TENSPED MO#Seri... -> 250 frames
[val] 2025#10#18#CANOTTIERI ONGINA PC#FERRAMENTA ASTORI BS#Se... -> 250 frames
[val] 2025#10#18#KERAKOLL SASSUOLO MO#MOTTA&ROSSI ASOLA REMED... -> 250 frames
[test] 2025#09#11#KERAKOLL SASSUOLO MO#NTRANSPORTS VILLADORO M... -> 250 frames
[test] 2025#09#12#VERO VOLLEY MONZA MB#INSUBRIA VOLLEY MORNAGO... -> 250 frames
[test] 2025#09#19#ASD PALLAVOLO SARONNO VA#VERO VOLLEY MONZA M... -> 250 frames
[test] 2025#09#23#TEAM VOLLEY CAZZA

## 4b. Ball frame extraction (train videos)

Same **`train/`** videos as player dataset, but **denser** sampling (`BALL_FRAME_INTERVAL`) for small/fast ball.


In [10]:
# Re-use player frame folders; copy subset with denser ball sampling into dataset_ball staging
BALL_FRAMES_DIR = FRAMES_DIR / 'ball'
for split in ['train', 'val', 'test']:
    (BALL_FRAMES_DIR / split).mkdir(parents=True, exist_ok=True)

ball_extract_log = []
ball_plan = [
    ('train', TRAIN_VIDEOS),
    ('val', VAL_VIDEOS),
    ('test', TEST_VIDEOS),
]
for split, vids in ball_plan:
    for v in vids:
        out_dir = BALL_FRAMES_DIR / split / re.sub(r'[^\w\-]+', '_', v.stem)[:80]
        n = extract_frames(v, out_dir, frame_interval=BALL_FRAME_INTERVAL, max_frames=MAX_FRAMES_PER_VIDEO)
        ball_extract_log.append({'split': split, 'video': v.name, 'frames_saved': n})
        print(f'[ball {split}] {v.name[:50]}... -> {n} frames')

pd.DataFrame(ball_extract_log).to_csv(OUTPUT_DIR / 'ball_frame_extraction_log.csv', index=False)


[ball train] 2025#09#23#VERO VOLLEY MONZA MB#ASD PALLAVOLO SARO... -> 250 frames
[ball train] 2025#09#25#NTRANSPORTS VILLADORO MO#MORE VOLLEY MO... -> 250 frames
[ball train] 2025#09#26#CANOTTIERI ONGINA PC#ASD PALLAVOLO CREM... -> 250 frames
[ball train] 2025#10#11#MOTTA&ROSSI ASOLA REMEDELLO MN#VERO VOL... -> 250 frames
[ball train] 2025#10#11#PALLAV CISANO BSCO SSDARL BG#CANOTTIERI... -> 250 frames
[ball train] 2025#10#11#PALLAV CISANO BSCO SSDARL BG#CANOTTIERI... -> 250 frames
[ball train] 2025#10#11#ZOTUP SCANZOROSCIATE BG#BECA TENSPED MO... -> 250 frames
[ball val] 2025#10#18#CANOTTIERI ONGINA PC#FERRAMENTA ASTORI ... -> 250 frames
[ball val] 2025#10#18#KERAKOLL SASSUOLO MO#MOTTA&ROSSI ASOLA ... -> 250 frames
[ball test] 2025#09#11#KERAKOLL SASSUOLO MO#NTRANSPORTS VILLAD... -> 250 frames
[ball test] 2025#09#12#VERO VOLLEY MONZA MB#INSUBRIA VOLLEY MO... -> 250 frames
[ball test] 2025#09#19#ASD PALLAVOLO SARONNO VA#VERO VOLLEY MO... -> 250 frames
[ball test] 2025#09#23#TEAM VOLLEY 

## 5. Pseudo-label players (bootstrap annotations)

You do not have manual YOLO labels yet. We bootstrap with a **pretrained YOLO person detector**, keep only high-confidence boxes as `player` labels, then fine-tune.

Per the PDF: auto-labels should be spot-checked. Section 7 visualizes them so you can delete bad frames before training if needed.

In [11]:
# Load small pretrained detector (person = COCO class 0)
try:
    teacher = YOLO(PRETRAINED_MODEL)
except Exception:
    print(f"Could not load {PRETRAINED_MODEL}, falling back to yolo11n.pt")
    PRETRAINED_MODEL = "yolo11n.pt"
    teacher = YOLO(PRETRAINED_MODEL)

SPLIT_MAP = {
    "train": (TRAIN_IMAGES, TRAIN_LABELS),
    "val":   (VAL_IMAGES, VAL_LABELS),
    "test":  (TEST_IMAGES, TEST_LABELS),
}

def clear_dir(d: Path):
    d.mkdir(parents=True, exist_ok=True)   # create the folder if missing
    for f in d.iterdir():
        if f.is_file():
            f.unlink()

# Fresh dataset dirs
for img_d, lbl_d in SPLIT_MAP.values():
    clear_dir(img_d)
    clear_dir(lbl_d)

def xyxy_to_yolo(x1, y1, x2, y2, w, h):
    bw, bh = x2 - x1, y2 - y1
    xc, yc = x1 + bw / 2, y1 + bh / 2
    return xc / w, yc / h, bw / w, bh / h

pseudo_stats = defaultdict(int)

for split, (img_dir, lbl_dir) in SPLIT_MAP.items():
    src_root = FRAMES_DIR / split
    if not src_root.exists():
        continue
    images = [p for p in src_root.rglob("*") if p.suffix.lower() in IMAGE_EXTS]
    print(f"Pseudo-labeling {split}: {len(images)} frames")
    for img_path in images:
        results = teacher.predict(
            source=str(img_path), conf=CONF_PSEUDO, classes=[PERSON_CLASS_ID],
            verbose=False, device=DEVICE, imgsz=IMAGE_SIZE
        )
        r0 = results[0]
        im = cv2.imread(str(img_path))
        if im is None:
            continue
        h, w = im.shape[:2]
        lines = []
        if r0.boxes is not None and len(r0.boxes):
            for box in r0.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                if (x2 - x1) < 20 or (y2 - y1) < 40:
                    continue
                xc, yc, bw, bh = xyxy_to_yolo(x1, y1, x2, y2, w, h)
                if bw <= 0 or bh <= 0 or bw > 0.9 or bh > 0.95:
                    continue
                lines.append(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
                pseudo_stats[f"{split}_boxes"] += 1
        dst_img = img_dir / img_path.name
        shutil.copy2(img_path, dst_img)
        (lbl_dir / f"{img_path.stem}.txt").write_text(
            "\n".join(lines) + ("\n" if lines else ""), encoding="utf-8"
        )
        pseudo_stats[f"{split}_images"] += 1

print(dict(pseudo_stats))


Pseudo-labeling train: 1750 frames
Pseudo-labeling val: 500 frames
Pseudo-labeling test: 2500 frames
{'train_boxes': 8783, 'train_images': 1750, 'val_boxes': 2703, 'val_images': 500, 'test_boxes': 14466, 'test_images': 2500}


## 5b. Pseudo-label ball (court ROI filtered)

Bootstrap ball boxes from COCO **sports-ball**; boxes outside court margins are dropped (Phase 1).


In [12]:
ball_teacher = YOLO(PRETRAINED_MODEL)
BALL_SPLIT_MAP = {
    'train': (BALL_TRAIN_IMAGES, BALL_TRAIN_LABELS),
    'val':   (BALL_VAL_IMAGES, BALL_VAL_LABELS),
    'test':  (BALL_TEST_IMAGES, BALL_TEST_LABELS),
}

def court_ok(cx, cy, w, h, mx=0.05, myt=0.08, myb=0.05):
    return (w * mx <= cx <= w * (1 - mx)) and (h * myt <= cy <= h * (1 - myb))

for img_d, lbl_d in BALL_SPLIT_MAP.values():
    clear_dir(img_d)
    clear_dir(lbl_d)

ball_stats = defaultdict(int)
for split, (img_dir, lbl_dir) in BALL_SPLIT_MAP.items():
    src_root = BALL_FRAMES_DIR / split
    if not src_root.exists():
        continue
    images = [p for p in src_root.rglob('*') if p.suffix.lower() in IMAGE_EXTS]
    print(f'Ball pseudo-label {split}: {len(images)} frames')
    for img_path in images:
        im = cv2.imread(str(img_path))
        if im is None:
            continue
        h, w = im.shape[:2]
        results = ball_teacher.predict(
            source=str(img_path), conf=BALL_CONF_PSEUDO, classes=[SPORTS_BALL_CLASS],
            verbose=False, device=DEVICE, imgsz=IMAGE_SIZE,
        )
        r0 = results[0]
        lines = []
        if r0.boxes is not None:
            for box in r0.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                area = (x2 - x1) * (y2 - y1)
                if area > w * h * 0.012:
                    continue
                if not court_ok(cx, cy, w, h):
                    continue
                xc, yc, bw, bh = xyxy_to_yolo(x1, y1, x2, y2, w, h)
                lines.append(f'0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
                ball_stats[f'{split}_boxes'] += 1
        shutil.copy2(img_path, img_dir / img_path.name)
        (lbl_dir / f'{img_path.stem}.txt').write_text('\n'.join(lines) + ('\n' if lines else ''), encoding='utf-8')
        ball_stats[f'{split}_images'] += 1

print(dict(ball_stats))


Ball pseudo-label train: 1750 frames
Ball pseudo-label val: 500 frames
Ball pseudo-label test: 2500 frames
{'train_images': 1750, 'train_boxes': 103, 'val_images': 500, 'val_boxes': 28, 'test_images': 2500, 'test_boxes': 331}


## 6. Validate YOLO dataset


In [13]:
NUM_CLASSES = 1  # player

def validate_yolo_split(image_dir, label_dir, num_classes=1):
    image_dir, label_dir = Path(image_dir), Path(label_dir)
    images = [p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS]
    missing, invalid, instances, corrupted = [], [], 0, []
    for img in images:
        label = label_dir / f"{img.stem}.txt"
        if not label.exists():
            missing.append(img.name)
            continue
        if cv2.imread(str(img)) is None:
            corrupted.append(img.name)
            continue
        for line_no, line in enumerate(label.read_text(encoding="utf-8").splitlines(), 1):
            if not line.strip():
                continue
            parts = line.split()
            if len(parts) != 5:
                invalid.append((label.name, line_no, "len!=5"))
                continue
            try:
                cls, xc, yc, bw, bh = map(float, parts)
            except ValueError:
                invalid.append((label.name, line_no, "non-numeric"))
                continue
            if int(cls) != cls or not (0 <= int(cls) < num_classes):
                invalid.append((label.name, line_no, "bad class"))
            if not all(0 <= x <= 1 for x in [xc, yc, bw, bh]):
                invalid.append((label.name, line_no, "oob"))
            if bw <= 0 or bh <= 0:
                invalid.append((label.name, line_no, "nonpos"))
            instances += 1
    return {
        "images": len(images), "missing_labels": len(missing),
        "invalid_labels": len(invalid), "player_instances": instances,
        "corrupted_images": len(corrupted),
    }

validation_results = {}
for split, img_d, lbl_d in [
    ("train", TRAIN_IMAGES, TRAIN_LABELS),
    ("val", VAL_IMAGES, VAL_LABELS),
    ("test", TEST_IMAGES, TEST_LABELS),
]:
    validation_results[split] = validate_yolo_split(img_d, lbl_d, NUM_CLASSES)
    print(split, validation_results[split])


train {'images': 1750, 'missing_labels': 0, 'invalid_labels': 0, 'player_instances': 8815, 'corrupted_images': 0}
val {'images': 500, 'missing_labels': 0, 'invalid_labels': 0, 'player_instances': 2708, 'corrupted_images': 0}
test {'images': 2500, 'missing_labels': 0, 'invalid_labels': 0, 'player_instances': 14488, 'corrupted_images': 0}


## 7. Visualize ground-truth (pseudo) labels - spot-check!


In [14]:
def draw_yolo_labels(image_path, label_path, class_names={0: "player"}):
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if Path(label_path).exists():
        for line in Path(label_path).read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            cls, xc, yc, bw, bh = map(float, line.split())
            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w - 1, x2), min(h - 1, y2)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                img, class_names.get(int(cls), str(int(cls))),
                (x1, max(20, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2
            )
    return img

def show_annotated_samples(image_dir, label_dir, n=6):
    images = [p for p in Path(image_dir).iterdir() if p.suffix.lower() in IMAGE_EXTS]
    with_boxes = []
    for p in images:
        t = (Path(label_dir) / f"{p.stem}.txt").read_text(encoding="utf-8").strip()
        if t:
            with_boxes.append(p)
    pool = with_boxes or images
    sample = pool[: min(n, len(pool))]
    if not sample:
        print("No images found.")
        return
    rows = int(np.ceil(len(sample) / 2))
    plt.figure(figsize=(14, 5 * rows))
    for i, img_path in enumerate(sample, 1):
        annotated = draw_yolo_labels(img_path, Path(label_dir) / f"{img_path.stem}.txt")
        plt.subplot(rows, 2, i)
        plt.imshow(annotated)
        plt.axis("off")
        plt.title(img_path.name[:40])
    plt.tight_layout()
    plt.show()

show_annotated_samples(TRAIN_IMAGES, TRAIN_LABELS)


## 8. Create `data.yaml`


In [8]:
data_yaml = {
    "path": str(DATASET_ROOT).replace("\\", "/"),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "player"},
}
DATA_YAML = PROJECT_ROOT / "data.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)
print(DATA_YAML.read_text(encoding="utf-8"))


path: /media/iflair/New Volume/VolleyScout/dataset
train: images/train
val: images/val
test: images/test
names:
  0: player



## 8b. Create `data_ball.yaml`


In [7]:
ball_data_yaml = {
    'path': str(BALL_DATASET_ROOT).replace('\\', '/'),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {0: 'ball'},
}
with open(DATA_BALL_YAML, 'w', encoding='utf-8') as f:
    yaml.safe_dump(ball_data_yaml, f, sort_keys=False)
print(DATA_BALL_YAML.read_text(encoding='utf-8'))


path: /media/iflair/New Volume/VolleyScout/dataset_ball
train: images/train
val: images/val
test: images/test
names:
  0: ball



## 9. Pretrained baseline (before fine-tune)


In [6]:
baseline_model = YOLO(PRETRAINED_MODEL)
test_images = (
    sorted([p for p in TEST_IMAGES.iterdir() if p.suffix.lower() in IMAGE_EXTS])
    if TEST_IMAGES.exists() else []
)
if test_images:
    baseline_model.predict(
        source=str(test_images[0]), conf=0.25, save=True,
        project=str(RUNS_DIR), name="baseline_image", exist_ok=True
    )
    print("Baseline image inference ->", RUNS_DIR / "baseline_image")
else:
    print("No test images yet - continue after extraction.")



image 1/1 /media/iflair/New Volume/VolleyScout/dataset/images/test/2025_09_11_KERAKOLL_SASSUOLO_MO_NTRANSPORTS_VILLADORO_MO_Amichevole_01_Villadoro_f00000000.jpg: 384x640 11 persons, 1 tv, 7.5ms
Speed: 54.6ms preprocess, 7.5ms inference, 18.7ms postprocess per image at shape (1, 3, 384, 640)
Results saved to /media/iflair/New Volume/VolleyScout/runs/baseline_image
Baseline image inference -> /media/iflair/New Volume/VolleyScout/runs/baseline_image


## 10. Fine-tune YOLO on volleyball players (GTX 1650 settings)

Weights are copied to `models/` after training.


In [10]:
model = YOLO(PRETRAINED_MODEL)

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(RUNS_DIR),
    name="volleyball_player_detection",
    exist_ok=True,
    seed=SEED,
    plots=False,  # skip slow label plots on first run
    patience=12,
    workers=0,  # Windows: avoid dataloader hang
    cache=False,
    amp=True,          # mixed precision -> less VRAM
    cos_lr=True,
    close_mosaic=8,
)

print("Training finished.")


New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.164 🚀 Python-3.12.7 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050, 5804MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=8, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/media/iflair/New Volume/VolleyScout/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, 

In [11]:
# Locate best weights and copy locally into models/
run_weights = RUNS_DIR / "volleyball_player_detection" / "weights"
BEST_MODEL = run_weights / "best.pt"
LAST_MODEL = run_weights / "last.pt"

assert BEST_MODEL.exists(), f"best.pt not found at {BEST_MODEL}"

LOCAL_BEST = MODELS_DIR / "volleyball_player_best.pt"
LOCAL_LAST = MODELS_DIR / "volleyball_player_last.pt"
shutil.copy2(BEST_MODEL, LOCAL_BEST)
if LAST_MODEL.exists():
    shutil.copy2(LAST_MODEL, LOCAL_LAST)

meta = {
    "best": str(LOCAL_BEST),
    "pretrained": PRETRAINED_MODEL,
    "epochs": EPOCHS,
    "imgsz": IMAGE_SIZE,
    "batch": BATCH_SIZE,
    "classes": {"0": "player"},
    "events_supported": EVENT_NAMES,
}
(MODELS_DIR / "model_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("Saved weights:")
print(" ", LOCAL_BEST)
print(" ", LOCAL_LAST if LOCAL_LAST.exists() else "(no last.pt)")
print(" ", MODELS_DIR / "model_meta.json")


Saved weights:
  /media/iflair/New Volume/VolleyScout/models/volleyball_player_best.pt
  /media/iflair/New Volume/VolleyScout/models/volleyball_player_last.pt
  /media/iflair/New Volume/VolleyScout/models/model_meta.json


## 10b. Fine-tune YOLO on volleyball ball

Requires **§5b** ball labels. Saves `models/volleyball_ball_best.pt`.


In [11]:
ball_model = YOLO(PRETRAINED_MODEL)
ball_train_results = ball_model.train(
    data=str(DATA_BALL_YAML),
    epochs=BALL_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(RUNS_DIR),
    name='volleyball_ball_detection',
    exist_ok=True,
    seed=SEED,
    plots=False,
    patience=10,
    workers=0,
    cache=False,
    amp=True,
    cos_lr=True,
    close_mosaic=8,
)
print('Ball training finished.')


New https://pypi.org/project/ultralytics/8.4.166 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.164 🚀 Python-3.12.7 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050, 5804MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=8, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/sarthak/Music/VolleyScout/data_ball.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, 

In [12]:
ball_run = RUNS_DIR / 'volleyball_ball_detection' / 'weights'
BALL_BEST = ball_run / 'best.pt'
assert BALL_BEST.exists(), f'missing {BALL_BEST}'
shutil.copy2(BALL_BEST, LOCAL_BALL_BEST)
(MODELS_DIR / 'ball_model_meta.json').write_text(
    json.dumps({'best': str(LOCAL_BALL_BEST), 'epochs': BALL_EPOCHS, 'classes': {'0': 'ball'}}, indent=2),
    encoding='utf-8',
)
print('Ball weights:', LOCAL_BALL_BEST)


Ball weights: /home/sarthak/Music/VolleyScout/models/volleyball_ball_best.pt


## 11b. Evaluate ball detector


In [12]:
if LOCAL_BALL_BEST.exists() and any(BALL_TEST_IMAGES.glob('*')):
    bm = YOLO(str(LOCAL_BALL_BEST))
    ball_metrics = bm.val(data=str(DATA_BALL_YAML), split='test', device=DEVICE, imgsz=IMAGE_SIZE, verbose=False)
    print('Ball test mAP50:', float(ball_metrics.box.map50))
else:
    print('Missing ball weights or empty dataset_ball/images/test - run §4b/§5b first.')

Ultralytics 8.4.164 🚀 Python-3.12.7 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050, 5804MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 91.0±8.8 MB/s, size: 144.3 KB)
val: Scanning /media/iflair/New Volume/VolleyScout/dataset_ball/labels/test... 2500 images, 2229 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2500/2500 2.4Kit/s 1.0s0.0s
val: New cache created: /media/iflair/New Volume/VolleyScout/dataset_ball/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 157/157 11.9it/s 13.2s0.1s
                   all       2500        318      0.383      0.239       0.21      0.121
Speed: 0.5ms preprocess, 3.3ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /media/iflair/New Volume/VolleyScout/runs/detect/val-2
Ball test mAP50: 0.2096607152595084


## 11. Evaluate fine-tuned model (real metrics only)


In [13]:
# Ensure LOCAL_BEST is defined even if you re-run only this cell
LOCAL_BEST = MODELS_DIR / "volleyball_player_best.pt"
assert LOCAL_BEST.exists(), f"Missing weights: {LOCAL_BEST}"

best_model = YOLO(str(LOCAL_BEST))

# workers=0 avoids Windows dataloader hangs (same issue as training)
val_metrics = best_model.val(
    data=str(DATA_YAML), split="val", imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE, device=DEVICE, plots=False, workers=0
)
test_metrics = best_model.val(
    data=str(DATA_YAML), split="test", imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE, device=DEVICE, plots=False, workers=0
)

def summarize_metrics(name, m):
    out = {"split": name}
    try:
        out["precision"] = float(m.box.mp)
        out["recall"] = float(m.box.mr)
        out["mAP50"] = float(m.box.map50)
        out["mAP50_95"] = float(m.box.map)
    except Exception as e:
        out["error"] = str(e)
    print(name, out)
    return out

metrics_rows = [summarize_metrics("val", val_metrics), summarize_metrics("test", test_metrics)]
pd.DataFrame(metrics_rows).to_csv(OUTPUT_DIR / "detection_metrics.csv", index=False)
print("Saved:", OUTPUT_DIR / "detection_metrics.csv")
print("NOTE: training showed NaN losses / ~0.02 mAP. Video export uses pretrained yolo11n person detector for usable detections.")


Ultralytics 8.4.164 🚀 Python-3.12.7 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050, 5804MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1922.0±1662.7 MB/s, size: 155.9 KB)
val: Scanning /media/iflair/New Volume/VolleyScout/dataset/labels/val.cache... 500 images, 17 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 131.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 125/125 22.7it/s 5.5s0.0s
                   all        500       2708      0.897      0.885      0.946      0.764
Speed: 0.3ms preprocess, 3.9ms inference, 0.0ms loss, 0.8ms postprocess per image
Ultralytics 8.4.164 🚀 Python-3.12.7 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050, 5804MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.1 ms, read: 66.9±17.0 MB/s, size: 121.4 KB)
val: Scan

## 12. Jersey number OCR helper + event state machine

- **Jersey**: crop upper torso -> EasyOCR -> keep 1–2 digit numbers
- **Events**: ball-motion + player-contact heuristics (works without event labels; improve later with supervised labels)

**v3 updates:** Best-frame selection for OCR, jersey number range filter (rejects invalid numbers like 66, 76), and confirmed-track short-circuit to avoid re-running OCR on already-identified players.

In [14]:

# ─── Helper function for finding readable video files ─────────────────────
def readable_videos(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    exts = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in exts])

# ─── Sync weights from training runs if needed ───────────────────────────
for src, dst in [(RUN_BEST, LOCAL_BEST), (RUN_BALL_BEST, LOCAL_BALL_BEST)]:
    if not dst.exists() and src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print('Copied', src.name, '->', dst)

# ─── Build the pipeline ──────────────────────────────────────────────────
# Fixes applied vs. previous version:
#   • min_ball_speed lowered 250 → 30 px/s → Set/Attack/Block/Dig now detected
#   • _classify_touch rewritten with speed bands that match HD 25-fps footage
#   • jersey backfill: after full video scan, final OCR votes are merged back
#     into every event row (jersey_number no longer empty for early events)
#   • Single output CSV per video (11 columns only)
#
# v3 & v4 Jersey identity improvements:
#   • max_jersey_number=25: rejects OCR misreads like 66, 76, 49 (scoreboard text)
#   • Best-frame selection: OCR only runs on sharp, well-lit, large-enough crops
#   • Upper-body top 55% crop: removes leg/shoe sponsor numbers
#   • Anti-spam voting: consecutive identical reads count as single vote
#   • Tighter confirmation: jersey_vote_min=3 AND confirm_min_avg_conf=0.55
#   • Post-process track clustering: merges fragmented tracks for same player
from volleyball_analytics.pipeline import PipelineConfig, VolleyballAnalyticsPipeline, process_video, OUTPUT_CSV_COLUMNS

pipeline_cfg = PipelineConfig(
    project_root=PROJECT_ROOT,
    player_weights=PRETRAINED_MODEL,
    prefer_finetuned_player=LOCAL_BEST.exists(),
    prefer_finetuned_ball=LOCAL_BALL_BEST.exists(),
    finetuned_player=LOCAL_BEST,
    finetuned_ball=LOCAL_BALL_BEST,
    device=DEVICE,
    tracker='botsort.yaml',
    infer_stride=1,
    player_infer_stride=2,
    ocr_interval_sec=1.0,
    use_motion_fallback=True,
    ocr_on_cpu=(DEVICE == 'cpu'),
    crops_dir=OUTPUT_DIR / 'jersey_crops',
    min_ball_speed=25.0,
    contact_cooldown_sec=0.55,
    contact_dist_px=160.0,

    # ── v3/v4: Jersey identity improvements ──────────────────────────────────
    max_jersey_number=25,
    ocr_min_bbox_area=5000,
    ocr_blur_threshold=60.0,
    ocr_brightness_min=40.0,
    ocr_brightness_max=230.0,
    ocr_skip_confirmed=False,
    ocr_min_interval_sec=0.5,
    jersey_vote_min=3,
    confirm_min_avg_conf=0.55,
    ocr_min_vote_conf=0.50,
    cluster_max_time_gap_sec=5.0,
    cluster_max_bbox_gap_px=150.0,

    # ── v5: Event logic correctness improvements ──────────────────────────
    max_rally_sec=25.0,              # force-close rallies > 25 s
    ocr_min_player_height_px=100,    # skip OCR on tiny (< 100 px) crops
    serve_to_reception_window_sec=5.0,  # Dig→Reception window after Serve

    # ── v7: Fabio feedback — rally sync, in-rally events, jersey, MP4 ─────
    debug_overlay=False,
    serve_grace_sec=2.2,
    point_dead_ball_sec=1.4,
    inter_rally_lockout_sec=3.0,
    jersey_export_min_conf=0.50,
    jersey_dominance_max_frac=0.35,
    ball_predict_frames=10,
    contact_dist_mid_rally_px=180.0,
)

analytics_pipeline = VolleyballAnalyticsPipeline(pipeline_cfg)
print('Player model          :', 'fine-tuned' if LOCAL_BEST.exists() else 'COCO pretrained')
print('Ball model            :', 'fine-tuned' if LOCAL_BALL_BEST.exists() else 'COCO pretrained')
print('Device                :', DEVICE)
print('min_ball_speed        :', pipeline_cfg.min_ball_speed, 'px/s')
print('max_jersey_number     :', pipeline_cfg.max_jersey_number)
print('jersey_vote_min       :', pipeline_cfg.jersey_vote_min)
print('confirm_min_avg_conf  :', pipeline_cfg.confirm_min_avg_conf)
print('ocr_blur_threshold    :', pipeline_cfg.ocr_blur_threshold)
print('cluster_max_time_gap  :', pipeline_cfg.cluster_max_time_gap_sec, 's')
print('max_rally_sec         :', pipeline_cfg.max_rally_sec, 's')
print('ocr_min_player_h_px   :', pipeline_cfg.ocr_min_player_height_px, 'px')
print('serve_reception_win   :', pipeline_cfg.serve_to_reception_window_sec, 's')
print('serve_grace_sec       :', pipeline_cfg.serve_grace_sec, 's')
print('jersey_export_min_conf:', pipeline_cfg.jersey_export_min_conf)


Player model          : fine-tuned
Ball model            : fine-tuned
Device                : 0
min_ball_speed        : 30.0 px/s
max_jersey_number     : 25
jersey_vote_min       : 3
confirm_min_avg_conf  : 0.55
ocr_blur_threshold    : 60.0
cluster_max_time_gap  : 8.0 s
max_rally_sec         : 25.0 s
ocr_min_player_h_px   : 100 px
serve_reception_win   : 5.0 s


## 13. Upload & Process Video Analytics

Upload any video below to run event and player jersey detection.
- **Duration**: Set `MAX_SECONDS = 300` (5 minutes), `60` (1 minute), or `None` (full match).


In [4]:
# ==============================================================================
# SECTION 13: UPLOAD & PROCESS VIDEO
# ==============================================================================
import re, shutil
from pathlib import Path
from IPython.display import display
import pandas as pd

# Set processing duration in seconds (300 = 5 minutes; None = Full video)
# For Fabio review retest use MAX_SECONDS = 300 on the Villadoro holdout clip.
MAX_SECONDS = 300  # Change to None for full match

# Ensure uploads folder exists
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# HOW TO RUN YOUR VIDEO:
#   Option 1 (EASIEST): Drag & drop or paste your video(s) into:
#       C:\Users\sarth\Music\VolleyScout\uploads
#       Then simply re-run this cell!
#
#   Option 2: Paste the full file path to your video below:
VIDEO_PATH = ""   # e.g., r"C:\path\to\match.mp4" (leave empty to use uploads/ folder)
# ------------------------------------------------------------------------------

def process_target_video(video_file: Path):
    dest = Path(video_file)
    if not dest.exists():
        print(f"❌ Error: Video file not found: {dest}")
        return

    print(f"\n{'='*70}")
    print(f"▶ Processing video: {dest.name}")
    dur_str = f"{MAX_SECONDS} seconds ({MAX_SECONDS/60:.1f} mins)" if MAX_SECONDS else "Full video (no limit)"
    print(f"  Duration limit: {dur_str} | Output: {OUTPUT_DIR}")
    print(f"{'='*70}\n")

    # v7: output = <video_stem>_analytics.mp4 (H.264 + audio when source has it).
    # Intermediate AVI is converted then deleted automatically.
    out_vid = OUTPUT_DIR / f"{dest.stem}_analytics.mp4"
    info = process_video(
        analytics_pipeline, dest, out_vid,
        max_seconds=MAX_SECONDS, split_label="upload"
    )

    # Confirm AVI was removed and MP4 exists
    avi_left = out_vid.with_suffix(".avi")
    if avi_left.exists() and out_vid.exists():
        avi_left.unlink(missing_ok=True)
        print(f"  [v7] Removed leftover AVI: {avi_left.name}")
    print(f"  Output video path: {info.get('out_video', out_vid)}")

    ev_df = pd.read_csv(info["out_csv"])
    print("\n=================== RESULTS ===================")
    print("Event breakdown:")
    print(ev_df["event_type"].value_counts().to_string())
    jerseys_ok = ev_df["jersey_number"].replace("", pd.NA).notna().sum()
    print(f"\nJerseys assigned: {jerseys_ok}/{len(ev_df)} events")
    print(f"Input video      : {dest}")
    print(f"Output video     : {info.get('out_video', out_vid)}")
    print(f"Events CSV       : {info['out_csv']}")
    display(ev_df)

# Execute processing
if VIDEO_PATH and Path(VIDEO_PATH).exists():
    process_target_video(Path(VIDEO_PATH))
else:
    valid_exts = {".mp4", ".avi", ".mov", ".mkv"}
    upload_videos = [f for f in UPLOAD_DIR.iterdir() if f.suffix.lower() in valid_exts]

    if upload_videos:
        print(f"✅ Found {len(upload_videos)} video(s) in {UPLOAD_DIR}:\n")
        for v in upload_videos:
            process_target_video(v)
    else:
        print(f"📁 Upload Folder: {UPLOAD_DIR}\n")
        print("👉 Put your video(s) into this folder:")
        print(f"   {UPLOAD_DIR}\n")
        print("   Once you copy/paste your video file there, click 'Run Cell' (▶) to start processing!")
        
        # Also render widget for Colab if available
        try:
            import ipywidgets as widgets
            uploader = widgets.FileUpload(accept=".mp4,.avi,.mov,.mkv", multiple=False, description="Upload Video")
            display(uploader)
            def on_upload(change):
                items = uploader.value
                if not items: return
                if isinstance(items, (tuple, list)):
                    fname, fdata = items[0]["name"], items[0]["content"]
                else:
                    fname = list(items.keys())[0]
                    fdata = items[fname]["content"]
                saved = UPLOAD_DIR / fname
                saved.write_bytes(bytes(fdata))
                process_target_video(saved)
            uploader.observe(on_upload, names="value")
        except Exception:
            pass


✅ Found 1 video(s) in C:\Users\sarth\Music\VolleyScout\uploads:


▶ Processing video: 2025#09#11#KERAKOLL SASSUOLO MO#NTRANSPORTS VILLADORO MO#Amichevole#01_Villadoro_bmb_25.MP4
  Duration limit: Full video (no limit) | Output: C:\Users\sarth\Music\VolleyScout\output

  [v5] Jersey state reset for: 2025#09#11#KERAKOLL SASSUOLO MO#NTRANSPORTS VILLADORO MO#Amichevole#01_Villadoro_bmb_25.MP4
  frame 400/106573 (128s)
  frame 800/106573 (252s)
  frame 1200/106573 (373s)
  frame 1600/106573 (630s)
  frame 2000/106573 (755s)
  frame 2400/106573 (886s)
  frame 2800/106573 (1011s)
  frame 3200/106573 (1138s)
  frame 3600/106573 (1284s)
  frame 4000/106573 (1438s)
  frame 4400/106573 (1570s)
  frame 4800/106573 (1692s)
  frame 5200/106573 (1826s)
  frame 5600/106573 (1934s)
  frame 6000/106573 (1976s)
  frame 6400/106573 (2018s)
  frame 6800/106573 (2060s)
  frame 7200/106573 (2096s)
  frame 7600/106573 (2133s)
  frame 8000/106573 (2174s)
  frame 8400/106573 (2215s)
  frame 8800/106573 (2253s)


,event_id,video_name,rally_id,event_type,time_start_sec,time_end_sec,timestamp,player_track_id,jersey_number,jersey_confidence,event_confidence
0,1,2025#09#11#KERAKOLL SASSUOLO MO#NTRANSPORTS VI...,1,Rally,1.32,1.37,00:00:01,3.0,6.0,0.333,1.00
1,2,2025#09#11#KERAKOLL SASSUOLO MO#NTRANSPORTS VI...,1,Serve,1.32,4.40,00:00:01,3.0,6.0,0.333,0.85
2,3,2025#09#11#KERAKOLL SASSUOLO MO#NTRANSPORTS VI...,1,Point,4.40,4.45,00:00:04,NaN,NaN,NaN,1.00
3,4,2025#09#11#KERAKOLL SASSUOLO MO#NTRANSPORTS VI...,2,Rally,6.96,7.01,00:00:07,3.0,6.0,0.667,1.00
4,5,2025#09#11#KERAKOLL SASSUOLO MO#NTRANSPORTS VI...,2,Serve,6.96,8.52,00:00:07,3.0,6.0,0.667,0.80
...,...,...,...,...,...,...,...,...,...,...,...
1418,1419,2025#09#11#KERAKOLL SASSUOLO MO#NTRANSPORTS VI...,313,Point,4250.36,4250.41,01:10:50,NaN,NaN,NaN,1.00
1419,1420,2025#09#11#KERAKOLL SASSUOLO MO#NTRANSPORTS VI...,314,Rally,4253.96,4254.01,01:10:54,8500.0,5.0,0.635,1.00
1420,1421,2025#09#11#KERAKOLL SASSUOLO MO#NTRANSPORTS VI...,314,Serve,4253.96,4257.48,01:10:54,8500.0,5.0,0.635,0.80
1421,1422,2025#09#11#KERAKOLL SASSUOLO MO#NTRANSPORTS VI...,314,Set,4257.48,4262.92,01:10:57,8482.0,10.0,1.000,0.72


In [ ]:
# ─── §15. Full export via CLI script ─────────────────────────────────────────
# Run from terminal / Colab shell:
#
#   Smoke (90 s per video):
#     !python export_analytics.py --max-seconds 90 --limit-videos 1
#
#   Full match(es) in holdout/:
#     !python export_analytics.py --full
#
#   Single specific video:
#     !python export_analytics.py --full --video /path/to/match.MP4
#
#   All videos in a custom folder:
#     !python export_analytics.py --full --video-dir /path/to/folder/
#
# Output per video:
#   output/<video_stem>_analytics.mp4          annotated H.264 MP4 (+audio if source has it)
#   output/<video_stem>_events.csv               single CSV (11 columns) — default
#   output/<video_stem>_jerseys.json             track_id -> jersey_number
#
# Optional merged CSV (not written for a single-video run unless requested):
#   python export_analytics.py ... --write-merged-csv
#
# Fabio 5-min retest:
#   python export_analytics.py --video "holdout/2025#09#11#KERAKOLL SASSUOLO MO#NTRANSPORTS VILLADORO MO#Amichevole#01_Villadoro_bmb_25.MP4" --max-seconds 300 --debug-overlay
#   python -m volleyball_analytics.evaluation --csv "output/..._events.csv" --duration-sec 300
#
# CSV columns: event_id | video_name | rally_id | event_type |
#              time_start_sec | time_end_sec | timestamp |
#              player_track_id | jersey_number | jersey_confidence | event_confidence


In [ ]:
# CLI: smoke or full holdout export (uses fine-tuned player + ball when present)
# !python export_analytics.py --max-seconds 90 --limit-videos 1 --split holdout
# !python export_analytics.py --full --split holdout
print('Uncomment a line above, or use §13b for in-notebook holdout batch.')


## 15. Error analysis samples


In [ ]:
best_model = YOLO(str(LOCAL_BEST))
test_images = sorted([p for p in TEST_IMAGES.iterdir() if p.suffix.lower() in IMAGE_EXTS])
if test_images:
    best_model.predict(
        source=[str(p) for p in test_images[:6]],
        conf=CONF_DET, save=True,
        project=str(RUNS_DIR), name="error_analysis_samples", exist_ok=True
    )
    print("Saved prediction samples ->", RUNS_DIR / "error_analysis_samples")
else:
    print("No test images.")


## 16. Project summary & deliverables

### What this notebook produces
| Deliverable | Location |
|-------------|----------|
| `data.yaml` | project root |
| Best / last weights | `models/volleyball_player_best.pt` |
| Train runs / curves | `runs/volleyball_player_detection/` |
| Detection metrics | `output/detection_metrics.csv` |
| Full annotated videos | `output/*_analytics.mp4` |
| Event timelines | `output/events_csv/` |
| Jersey maps | `output/*_jerseys.json` |

### Events covered
Serve · Reception · Set · Attack · Block · Dig · Rally · Point

Event spotting is **heuristic** (ball motion + player contact) because match videos have no event labels. For production accuracy, annotate event timestamps and train a temporal classifier - the CSV outputs are the right place to start labeling.

### Jersey numbers
OCR on torso crops; blur / occlusion / rear views will miss numbers. Improve by collecting cropped-digit labels.

### Checklist
- [x] Video-level equal train/test split (10/10)
- [x] Frame extraction + pseudo-label bootstrap
- [x] Fine-tune YOLO (local weights)
- [x] Val/test metrics
- [x] Full-video export to `output`
- [x] Tracking + jersey OCR + event overlay


## 17. Labeling and smoke QA

Create review manifests for manual ball and jersey-number annotation. Pseudo-labels are not ground truth: correct labels before training.


In [ ]:
# Export ball-labeling frames from TRAIN videos. Label ball boxes manually in YOLO format.
LABEL_REVIEW_DIR = PROJECT_ROOT / 'label_review'
BALL_REVIEW_DIR = LABEL_REVIEW_DIR / 'ball_frames'
BALL_REVIEW_DIR.mkdir(parents=True, exist_ok=True)
review_rows = []
for video in TRAIN_VIDEOS:
    cap = cv2.VideoCapture(str(video))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    for frame_idx in range(0, min(total, int(fps * 180)), max(1, int(fps * 1.5))):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()
        if not ok: continue
        name = f'{video.stem[:45]}_f{frame_idx:06d}.jpg'
        path = BALL_REVIEW_DIR / name
        cv2.imwrite(str(path), frame)
        review_rows.append({'image': str(path.relative_to(PROJECT_ROOT)), 'video': video.name,
                            'frame_idx': frame_idx, 'time_sec': round(frame_idx / fps, 3),
                            'ball_visible': '', 'label_status': 'unlabeled'})
    cap.release()
ball_manifest = LABEL_REVIEW_DIR / 'ball_annotation_manifest.csv'
pd.DataFrame(review_rows).to_csv(ball_manifest, index=False)
print('Ball annotation manifest:', ball_manifest, '| frames:', len(review_rows))


In [ ]:
# Jersey-number-region YOLO dataset skeleton. Add reviewed player crops and one `jersey_number` box per image.
JERSEY_DATASET_ROOT = PROJECT_ROOT / 'dataset_jersey_number'
for split in ('train', 'val', 'test'):
    (JERSEY_DATASET_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (JERSEY_DATASET_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)
jersey_yaml = PROJECT_ROOT / 'data_jersey_number.yaml'
with open(jersey_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump({'path': str(JERSEY_DATASET_ROOT), 'train': 'images/train', 'val': 'images/val',
                    'test': 'images/test', 'names': {0: 'jersey_number'}}, f, sort_keys=False)
print('After annotation: YOLO('yolo11n.pt').train(data=str(jersey_yaml), epochs=40, imgsz=640, device=DEVICE)')
print('Copy the best weight to:', MODELS_DIR / 'jersey_number_best.pt')


In [ ]:
# ─── §17. Smoke QA — inspect output CSV ──────────────────────────────────────
import pandas as pd
from pathlib import Path

csv_glob = sorted(OUTPUT_DIR.glob('*_events.csv'))
if not csv_glob:
    print('No events CSV found. Run §13 or §13b first.')
else:
    for csv_path in csv_glob:
        df = pd.read_csv(csv_path)
        print(f'\n=== {csv_path.name} ===')
        print(f'Total events : {len(df)}')
        print('Event types  :')
        print(df['event_type'].value_counts().to_string())
        jerseys_ok = df['jersey_number'].replace('', pd.NA).notna().sum()
        print(f'Jersey filled: {jerseys_ok}/{len(df)}')
        all_types = {'Serve','Reception','Set','Attack','Block','Dig','Rally','Point'}
        missing_types = all_types - set(df['event_type'].unique())
        if missing_types:
            print(f'WARNING — missing event types: {sorted(missing_types)}')
        else:
            print('OK — all 8 event types present')
        display(df)
